In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    E_z = np.exp(Z)
    return E_z / np.sum(E_z, axis=1, keepdims=True)


def train(X, labels, K, lr=0.1, epochs=500):
    m, i = X.shape
    W = np.zeros((K, i))
    b = np.zeros(K)
    Y = np.eye(K)[labels]       
    ##eye는 K X K 행렬이고 Lables에 있는 정수는 행마다 one hot
    #encoding의 위치를 출력 (m, K) one-hot

    history = []

    for ep in range(epochs):
        # forward (루프 안!)
        Z = X @ W.T + b            # (m, K)
        P = softmax(Z)             # (m, K)

        # loss
        loss = -np.sum(Y * np.log(P + 1e-12)) / m
        history.append(loss)

        # backward
        # backward is always same in GLM
        dW = (P - Y).T @ X / m     # (K, i)
        db = (P - Y).mean(axis=0)  # (K,)
        assert dW.shape == W.shape and db.shape == b.shape

        # update
        W = W - lr * dW
        b = b - lr * db

    return W, b, history

def train_batch(X,batch_size, labels, K, lr=0.1, epochs=500,seed = 42):

    m, i = X.shape
    W = np.zeros((K, i))
    b = np.zeros(K)
    Y = np.eye(K)[labels]
    rng = np.random.default_rng(seed)

    history = []

    for epoch in range(epochs):

        idx = rng.permutation(m)

        for start in range(0,m,batch_size):

            batch = idx[start:start+batch_size]

            Xb = X[batch] #(B,i)
            Yb = Y[batch] #(B,K)
            
            B= len(batch)

            P = softmax(Xb @ W.T + b)             # (B, K)
            
            # backward
            # backward is always same in GLM
            dW = (P - Yb).T @ Xb / B     # (K, i)
            db = (P - Yb).mean(axis=0)  # (K,)
            assert dW.shape == W.shape and db.shape == b.shape

            #update
            W = W - lr * dW
            b = b - lr * db
            
        P_all = softmax(X @ W.T + b)
        history.append(-np.sum(Y * np.log(P_all + 1e-12)) / m)

    return W, b, history

def evaluate(X_test,labels_test,W,b):

    m = X_test.shape[0]
    Z =  X_test @ W.T + b
    P = softmax(Z)

    loss = -np.mean(np.log(P[np.arange(m), labels_test] + 1e-12))
    
    pred = P.argmax(axis=1)                        # (m,)
    acc = (pred == labels_test).mean()             # True/False의 평균 = 정답 비율

    return loss, acc

In [5]:
from sklearn.datasets import load_iris
data = load_iris()
X, labels = data.data, data.target        # (150, 4), K=3
X = (X - X.mean(axis=0)) / X.std(axis=0)  # 표준화

In [6]:
print(data.feature_names)
print(data.target_names)   # ['setosa' 'versicolor' 'virginica']

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
['setosa' 'versicolor' 'virginica']


In [7]:
import numpy as np

rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
X, labels = X[idx], labels[idx]
X_train, X_test = X[:120], X[120:]
y_train, y_test = labels[:120], labels[120:]

In [ ]:
K = 3
W1, b1, h1 = train(X_train, y_train, K, lr=0.1, epochs=200)
W2, b2, h2 = train_batch(X_train, batch_size=16, labels=y_train, K=K, lr=0.1, epochs=200)

for name, W, b in [("full", W1, b1), ("mini", W2, b2)]:
    tr = evaluate(X_train, y_train, W, b)
    te = evaluate(X_test, y_test, W, b)
    print(f"{name}: train loss {tr[0]:.3f} acc {tr[1]:.3f} | test loss {te[0]:.3f} acc {te[1]:.3f}")

plt.plot(h1, label="full batch"); plt.plot(h2, label="mini-batch 16")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()